# Problem Set 4. Text III: attention & transformers

**Course:** Analysis of Unstructured Data  
**Instructor:** Janusz Szwabiński  
**Goal:** Understand and reproduce the attention mechanism, then use a pretrained transformer.  

---


In [ ]:
# Setup environment & imports
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


### Problem 1 [Theory] (Scaled dot-product attention)

**Task:**
Derive scaled dot-product attention $\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$. Give the variance argument for the $1/\sqrt{d_k}$ scaling (what happens to the softmax as $d_k$ grows without it?).

---

### Assumptions & Notation

1. **Input Matrices:**
   - Query matrix: $Q \in \mathbb{R}^{n \times d_k}$ with row vectors $q_i \in \mathbb{R}^{d_k}$ for $i \in \{1, \dots, n\}$.
   - Key matrix: $K \in \mathbb{R}^{m \times d_k}$ with row vectors $k_j \in \mathbb{R}^{d_k}$ for $j \in \{1, \dots, m\}$.
   - Value matrix: $V \in \mathbb{R}^{m \times d_v}$ with row vectors $v_j \in \mathbb{R}^{d_v}$ for $j \in \{1, \dots, m\}$.
   Here $n$ denotes query sequence length, $m$ denotes key/value sequence length ($n = m$ in self-attention), $d_k$ denotes the projection dimension of queries and keys, and $d_v$ denotes the value dimension.

2. **Probabilistic Modeling of Representation Vectors:**
   Assume the individual feature components $q_{i,l}$ and $k_{j,l}$ ($l \in \{1, \dots, d_k\}$) are independent, identically distributed (i.i.d.) random variables with zero mean and unit variance:
   $$\mathbb{E}[q_{i,l}] = 0, \quad \mathbb{E}[k_{j,l}] = 0$$
   $$\mathrm{Var}(q_{i,l}) = 1, \quad \mathrm{Var}(k_{j,l}) = 1$$
   Furthermore, queries and keys are mutually independent: $\mathbb{E}[q_{i,l} k_{j, l'}] = 0$ for all $l, l'$.

3. **Softmax Operator:**
   For any logit vector $z = (z_1, \dots, z_m)^\top \in \mathbb{R}^{m}$, the row-wise softmax operator $\mathrm{softmax}: \mathbb{R}^m \to \Delta^{m-1}$ maps $z$ onto the probability simplex:
   $$\mathrm{softmax}(z)_j = \frac{\exp(z_j)}{\sum_{l=1}^m \exp(z_l)}, \quad j \in \{1, \dots, m\}$$

---

### Formal Proposition

1. **Scaled Dot-Product Attention:**
   The output representation for query sequence $Q$ interacting with key-value pairs $(K, V)$ is given by:
   $$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$
   where the $i$-th row is the convex combination:
   $$\mathrm{Attn}(Q, K, V)_i = \sum_{j=1}^m A_{ij} v_j, \quad A_{ij} = \frac{\exp\left(\frac{q_i k_j^\top}{\sqrt{d_k}}\right)}{\sum_{l=1}^m \exp\left(\frac{q_i k_l^\top}{\sqrt{d_k}}\right)}$$

2. **Variance of Unscaled Dot Products:**
   The unscaled affinity score $S_{ij} = q_i k_j^\top = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$ has mean zero and variance proportional to the key dimension $d_k$:
   $$\mathbb{E}[S_{ij}] = 0, \quad \mathrm{Var}(S_{ij}) = d_k, \quad \sigma(S_{ij}) = \sqrt{d_k}$$

3. **Variance Normalization via $1/\sqrt{d_k}$:**
   The scaled logit $Z_{ij} = \frac{S_{ij}}{\sqrt{d_k}}$ satisfies:
   $$\mathbb{E}[Z_{ij}] = 0, \quad \mathrm{Var}(Z_{ij}) = 1$$
   independent of the projection dimension $d_k$.

4. **Softmax Degeneracy and Gradient Vanishing Without Scaling:**
   As $d_k \to \infty$, the variance of unscaled logits $S_{ij}$ diverges as $O(d_k)$. The gap between the maximal logit and other logits grows as $O(\sqrt{d_k \log m})$, driving the softmax distribution to a one-hot distribution (hard $\mathrm{argmax}$). In this saturated regime, the Jacobian $\frac{\partial \mathrm{softmax}(z)_j}{\partial z_l} = s_j (\delta_{jl} - s_l)$ approaches zero exponentially, causing vanishing gradients and stalling optimization.

---

### Step-by-Step Derivation

#### Step 1: Attention as Content-Based Memory Retrieval
In memory retrieval, a query $q_i$ seeks relevant information stored in keys $k_j$ to retrieve corresponding values $v_j$. 
The bilinear affinity between $q_i$ and $k_j$ is given by the inner product:
$$S_{ij} = \langle q_i, k_j \rangle = q_i k_j^\top = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$$
Collecting all pairs into matrix notation:
$$S = Q K^\top \in \mathbb{R}^{n \times m}$$
To form normalized attention weights summing to 1 across keys for each query row, we apply the row-wise softmax:
$$A = \mathrm{softmax}(S)$$
The retrieved values are then the weighted sum:
$$\mathrm{Attn}(Q, K, V) = A V = \mathrm{softmax}(Q K^\top) V$$

#### Step 2: Expectation and Variance of the Unscaled Dot Product
We analyze the distribution of each scalar logit $S_{ij} = \sum_{l=1}^{d_k} q_{i,l} k_{j,l}$.
Using linearity of expectation and independence between components:
$$\mathbb{E}[S_{ij}] = \mathbb{E}\left[ \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \right] = \sum_{l=1}^{d_k} \mathbb{E}[q_{i,l}] \mathbb{E}[k_{j,l}] = \sum_{l=1}^{d_k} 0 \cdot 0 = 0$$

For the variance, because the terms $q_{i,l} k_{j,l}$ are independent across dimension index $l \in \{1, \dots, d_k\}$:
$$\mathrm{Var}(S_{ij}) = \mathrm{Var}\left( \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \right) = \sum_{l=1}^{d_k} \mathrm{Var}(q_{i,l} k_{j,l})$$

For each individual term:
$$\mathrm{Var}(q_{i,l} k_{j,l}) = \mathbb{E}[(q_{i,l} k_{j,l})^2] - (\mathbb{E}[q_{i,l} k_{j,l}])^2 = \mathbb{E}[q_{i,l}^2] \mathbb{E}[k_{j,l}^2] - 0$$
Since $\mathbb{E}[q_{i,l}] = 0$ and $\mathrm{Var}(q_{i,l}) = 1$, we have $\mathbb{E}[q_{i,l}^2] = \mathrm{Var}(q_{i,l}) + (\mathbb{E}[q_{i,l}])^2 = 1 + 0 = 1$. Similarly $\mathbb{E}[k_{j,l}^2] = 1$.
Therefore:
$$\mathrm{Var}(q_{i,l} k_{j,l}) = 1 \cdot 1 = 1$$
Summing over all $d_k$ dimensions:
$$\mathrm{Var}(S_{ij}) = \sum_{l=1}^{d_k} 1 = d_k$$
The standard deviation of the unscaled inner product is therefore:
$$\sigma(S_{ij}) = \sqrt{\mathrm{Var}(S_{ij})} = \sqrt{d_k}$$

#### Step 3: Scaling by $1/\sqrt{d_k}$
To keep the scale of the logits invariant to the choice of projection dimension $d_k$, we define the scaled logit:
$$Z_{ij} = \frac{S_{ij}}{\sqrt{d_k}}$$
Its expectation is:
$$\mathbb{E}[Z_{ij}] = \mathbb{E}\left[ \frac{S_{ij}}{\sqrt{d_k}} \right] = \frac{1}{\sqrt{d_k}} \mathbb{E}[S_{ij}] = 0$$
Using the scaling property of variance $\mathrm{Var}(a X) = a^2 \mathrm{Var}(X)$:
$$\mathrm{Var}(Z_{ij}) = \mathrm{Var}\left( \frac{S_{ij}}{\sqrt{d_k}} \right) = \left(\frac{1}{\sqrt{d_k}}\right)^2 \mathrm{Var}(S_{ij}) = \frac{1}{d_k} \cdot d_k = 1$$
Furthermore, by the Lindeberg-Lévy Central Limit Theorem, as $d_k \to \infty$:
$$Z_{ij} = \frac{1}{\sqrt{d_k}} \sum_{l=1}^{d_k} q_{i,l} k_{j,l} \xrightarrow{d} \mathcal{N}(0, 1)$$
Thus, dividing by $\sqrt{d_k}$ standardizes the dot product to standard normal distribution $\mathcal{N}(0, 1)$ regardless of how large $d_k$ is.

#### Step 4: Softmax Behavior as $d_k$ Grows Without Scaling
What happens when we omit the $1/\sqrt{d_k}$ factor and compute $\mathrm{softmax}(S_{i,:})$ directly?

1. **Logit Magnitude Growth:**
   As $d_k$ grows (e.g. $d_k = 64, 512, 1024$), the standard deviation $\sigma(S_{ij}) = \sqrt{d_k}$ increases dramatically ($\sqrt{64}=8$, $\sqrt{512}\approx 22.6$, $\sqrt{1024}=32$).
   For $m$ keys, the expected difference between the largest logit $S_{(1)}$ and the runner-up $S_{(2)}$ scales as $O(\sqrt{d_k \log m})$.

2. **Saturation to Hard Argmax:**
   Let $j^* = \operatorname{argmax}_j S_{ij}$. For any competitor $j \ne j^*$:
   $$\frac{\exp(S_{ij})}{\exp(S_{ij^*})} = \exp(S_{ij} - S_{ij^*}) \le \exp(- c \sqrt{d_k}) \to 0 \quad \text{as } d_k \to \infty$$
   Hence, the softmax probabilities saturate:
   $$s_{j^*} \to 1, \quad s_j \to 0 \quad (\forall j \ne j^*)$$
   The continuous attention mechanism collapses into an inflexible hard selection operator.

3. **Gradient Vanishing in Backpropagation:**
   Let $s = \mathrm{softmax}(z) \in \mathbb{R}^m$. The Jacobian matrix $J \in \mathbb{R}^{m \times m}$ of the softmax transformation has elements:
   $$\frac{\partial s_j}{\partial z_l} = s_j (\delta_{jl} - s_l) = \begin{cases} s_j (1 - s_j), & \text{if } j = l \\ -s_j s_l, & \text{if } j \ne l \end{cases}$$
   When the distribution is saturated:
   - For the winner $j^*$: $s_{j^*} \approx 1 \implies s_{j^*}(1 - s_{j^*}) \approx 1 \cdot 0 = 0$.
   - For all other keys $j \ne j^*$: $s_j \approx 0 \implies s_j(1 - s_j) \approx 0 \cdot 1 = 0$.
   - For cross-terms ($j \ne l$): $-s_j s_l \approx 0$.
   Therefore:
   $$\lim_{d_k \to \infty} \frac{\partial s_j}{\partial S_{il}} = 0 \quad \forall j, l$$
   By the chain rule, gradients of the loss $\mathcal{L}$ with respect to queries and keys satisfy:
   $$\frac{\partial \mathcal{L}}{\partial q_i} = \sum_{j=1}^m \frac{\partial \mathcal{L}}{\partial A_{ij}} \frac{\partial A_{ij}}{\partial q_i} \to 0$$
   The gradients vanish exponentially with respect to $\sqrt{d_k}$, halting backpropagation through the query and key projection weights. The $1/\sqrt{d_k}$ scaling maintains variance at 1, placing the logits firmly inside the sensitive, high-gradient regime of the softmax function.

---

### Sanity Checks & Edge Cases

1. **Dimensional Consistency ($d_k = 1$):**
   When $d_k = 1$, $\sqrt{d_k} = 1$. The scaling factor is $\frac{1}{\sqrt{1}} = 1$. The scalar product $q_i k_j$ already has unit variance $\mathrm{Var}(q_i k_j) = 1 \cdot 1 = 1$, and no scaling is required.
2. **Orthogonal Queries and Keys:**
   If $q_i \perp k_j$ for all $j$, $S_{ij} = 0$. Then $\exp(0 / \sqrt{d_k}) = 1$, yielding uniform attention $A_{ij} = \frac{1}{m}$ for all $j$, independent of $d_k$.
3. **Temperature Analogy:**
   The scaling $1/\sqrt{d_k}$ is mathematically equivalent to setting the Boltzmann temperature parameter $T = \sqrt{d_k}$ in statistical physics: $p_j \propto \exp(E_j / T)$. As system degrees of freedom ($d_k$) increase, temperature must scale as $\sqrt{d_k}$ to preserve thermodynamic entropy and avoid crystallization into a single ground state.


### Problem 2 [Theory] (Permutation equivariance & Positional Encodings)

---
**Proof & Analysis:**


### Problem 3 [Theory] (Parameter count and asymptotic compute)

---
**Computation & Comparison:**


### Problem 4 [Python]

**Task:**
Implement scaled dot-product attention and multi-head attention from scratch and check your output against torch.nn.MultiheadAttention on random inputs (max abs. error < 10^-5).


In [ ]:
# Problem 4 [Python]
import torch
import torch.nn as nn

# TODO: Implement scaled dot-product & MultiheadAttention from scratch and verify against torch.nn.MultiheadAttention


### Problem 5 [Python]

**Task:**
Use a pretrained HuggingFace transformer two ways: (a) as a frozen sentence-embedding extractor feeding a linear classifier, and (b) fine-tuned end-to-end. Compare both against the TF-IDF baseline from Problem Set 3 on the same data.


In [ ]:
# Problem 5 [Python]
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch

# TODO: (a) Frozen sentence-embedding extractor + linear classifier
#       (b) Fine-tuned end-to-end
#       Compare with PS3 TF-IDF


### Problem 6 [Analysis]

---
**Visualisation & Interpretation:**
